In [1]:
# S.0 Import thư viện
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, element_at
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
import os, sys, glob, time, shutil

In [2]:
# S.0 Khởi tạo Spark
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
assert sys.version_info < (3, 12), (
    f"PySpark 3.5.1 chỉ chạy Python <= 3.11 (đang dùng {sys.version.split()[0]}). "
    "Hãy chọn interpreter/kernel Python 3.11.")

spark = SparkSession.builder.appName("AirbnbSuperhostStream").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [3]:
# S.0 Đường dẫn model
MODEL_PATH = "streaming/models/rf_superhost"
DATA_CSV = "../data/Listings.csv"
# MODEL_PATH = "hdfs://master-node:9000/AirbnbData/models/rf_superhost"
# DATA_CSV = "hdfs://master-node:9000/AirbnbData/Listings.csv"

SOURCE_DIR = "streaming/stream_source"   # nguồn đã chia thành file nhỏ
INPUT_DIR = "streaming/stream_input"     # thư mục Spark theo dõi luồng
os.makedirs(SOURCE_DIR, exist_ok=True)
os.makedirs(INPUT_DIR, exist_ok=True)

In [4]:
# S.1 Cột giữ lại cho luồng
KEEP = [
    "listing_id", "host_id", "city", "host_is_superhost",
    "price", "host_total_listings_count", "minimum_nights",
    "review_scores_rating", "review_scores_cleanliness", "review_scores_communication",
    "host_response_time", "instant_bookable", "host_identity_verified", "room_type",
]

In [5]:
# S.2 Chuẩn bị nguồn streaming (chạy 1 lần; có _SUCCESS nghĩa là đã ghi xong -> bỏ qua)
# Không inferSchema để khỏi quét toàn bộ file lớn -- readStream ở S.5 đã ép kiểu bằng SCHEMA.
if os.path.exists(SOURCE_DIR + "/_SUCCESS"):
    print("Đã có sẵn nguồn streaming -> bỏ qua S.2.")
else:
    raw = spark.read.csv(DATA_CSV, header=True, multiLine=True, escape='"', quote='"')
    mau = (raw.select(*KEEP)
              .dropna(subset=["host_is_superhost", "review_scores_rating"])
              .sample(False, 0.02, seed=42)   # lấy mẫu để có nhiều thành phố
              .limit(400))
    mau.repartition(20).write.option("header", True).mode("overwrite").csv(SOURCE_DIR)
    print("Đã chuẩn bị nguồn streaming (~400 listing, 20 file) trong:", SOURCE_DIR)

Đã có sẵn nguồn streaming -> bỏ qua S.2.


In [6]:
# S.3 Schema cho luồng (kiểu dữ liệu từng cột)
SCHEMA = StructType([
    StructField("listing_id", IntegerType(), True),
    StructField("host_id", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("host_is_superhost", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("host_total_listings_count", DoubleType(), True),
    StructField("minimum_nights", DoubleType(), True),
    StructField("review_scores_rating", DoubleType(), True),
    StructField("review_scores_cleanliness", DoubleType(), True),
    StructField("review_scores_communication", DoubleType(), True),
    StructField("host_response_time", StringType(), True),
    StructField("instant_bookable", StringType(), True),
    StructField("host_identity_verified", StringType(), True),
    StructField("room_type", StringType(), True),
])